# Stage 4, tier 3: full-transcript half-life and translation efficiency

The earlier stability comparison starts from coding sequences. This notebook asks what changes when inputs also include the untranslated regions surrounding the coding sequence, and whether adding a protein representation improves prediction. It introduces transcript datasets distributed through mRNABench; it does not add missing regions to the earlier BioLangFusion rows. Differences between those datasets' scores therefore cannot isolate the effect of including untranslated regions.

We first inspect whether the supplied sequences, coding-region annotations, targets and grouping information support the proposed comparisons. We then compare features calculated from different transcript regions. Finally, frozen encoders turn selected regions into numerical vectors, and ridge regression predicts each dataset's target from an individual vector or a concatenation. Here, “fusion” means joining pooled vectors before ridge regression, rather than training Phase A's token-level fusion heads.

Half-life, translation efficiency and ribosome load describe different outcomes. Their numerical labels remain separate throughout the analysis:

| Prediction target | Meaning and target construction here | Dataset keys |
| --- | --- | --- |
| RNA half-life | RNA persistence before degradation. These labels are first-principal-component summaries across 39 human or 27 mouse datasets, not half-life measurements in hours. | `rnahl-human`, `rnahl-mouse` |
| Translation efficiency | Translation activity relative to RNA abundance, estimated from ribosome profiling; not protein abundance. | `translation-efficiency-human`, `translation-efficiency-mouse` |
| Mean ribosome load | Ribosome load associated with a transcript isoform, kept distinct from half-life and translation efficiency. | `mrl-sugimoto` |
| Synthetic-mRNA half-life and ribosome load | Two separately named targets in the same supplied table; eligibility determines whether either is evaluated below. | `mrl-hl-lbkwk` |


The [Stage 4 guide](README.md#full-transcript-half-life-and-translation-efficiency) explains how these comparisons fit the broader experiments. The [dataset catalog and helpers](../shared_code/mrnabench.py) record the source and target construction for each table. [Research-plan](../../docs/Research_Project_Plan_Gantt.xlsx) `Gantt` and `Details` task 42 calls for feasibility checks before the half-life and translation-efficiency comparisons.

Part 1 inspects all configured tables. Part 2 evaluates calculated region features on eligible datasets. Part 3's `PRIMARY` setting selects the **two human datasets** for encoder comparisons; inspecting mouse datasets does not itself run mouse encoder replication.


---

## Setup

Prepare the environment using the root README's [notebook dependencies](../../README.md#notebook-dependencies) and select it as this notebook's kernel. Package installation belongs in that setup step.

### Imports


In [1]:
# Project paths, environment settings and elapsed-time reporting.
import os
import sys
import time
from pathlib import Path

# Enable CPU fallback for unsupported Apple GPU operations before importing torch.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Numerical arrays and dataset/result tables.
import numpy as np
import pandas as pd

# Encoder computation and hardware availability.
import torch

# Display width is measured in characters, not nucleotide positions.
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
print(
    "torch", torch.__version__,
    "| CUDA:", torch.cuda.is_available(),
    "| Apple GPU:", torch.backends.mps.is_available(),
)


torch 2.10.0+cu126 | CUDA: True | Apple GPU: False


### Project paths and shared code

Locate the project's `Code` directory before importing its dataset, feature, embedding and evaluation helpers. Cache and result locations are anchored to that directory, rather than depending on the kernel starting in `stage4`.


In [ ]:
# Find Code/shared_code from the repository root or any folder beneath it.
working_dir = Path.cwd().resolve()
for parent in (working_dir, *working_dir.parents):
    if (parent / "shared_code").is_dir():
        CODE_DIR = parent
        break
    if (parent / "Code" / "shared_code").is_dir():
        CODE_DIR = parent / "Code"
        break
else:
    raise FileNotFoundError(
        "Cannot locate Code/shared_code. Start this notebook from within the repository."
    )

# Make shared helpers importable without changing the kernel's working folder.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Transcript regions, dataset checks, cached embeddings and grouped ridge probes.
from shared_code import mrnabench

# Checkpoint, alphabet and context settings for the selected encoders.
from shared_code.encoders import ENCODERS


### Settings, file locations and hardware

`PRIMARY` selects the human half-life and translation-efficiency datasets for Part 3. `PROBE_SEEDS` contains the nine split-assignment seeds inherited from the benchmark procedure. Part 2 uses the first three; Part 3 uses all nine. These seeds change which groups are held out, not the pretrained encoder weights.

`RUN_EMBEDDINGS` controls both loading or computing the Part 3 matrices and the later encoder comparisons. With it enabled, compatible complete caches can be reused; missing matrices require encoder inference. Parts 1 and 2 can run independently of that switch, although Part 1 may download uncached dataset files.

The device preference is NVIDIA CUDA, then Apple MPS, then CPU. The notebook supplies no explicit precision override to the shared encoder loader. Dataset inspection and ridge fitting use CPU operations; encoder inference can use the selected accelerator. No measured runtime or peak-memory preflight is implemented here.


In [ ]:
# Keep the existing human encoder comparison and split-assignment sequence.
PRIMARY = ("rnahl-human", "translation-efficiency-human")
PROBE_SEEDS = (2541, 2547, 413, 412, 411, 321, 421, 2515, 2516)
RUN_EMBEDDINGS = True

# Raw tables, complete representation matrices and small result tables have distinct homes.
RESULTS_DIR = CODE_DIR / "stage4" / "results" / "halflife_te"
EMBED_DIR = CODE_DIR / "stage3" / "embeddings" / "mrnabench"
DATA_CACHE = mrnabench.DEFAULT_CACHE  # Repository .data-cache/mrnabench; ignored by Git.


In [8]:
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"device {DEVICE}; results in {RESULTS_DIR}, embeddings in {EMBED_DIR}")


device cuda; results in results\halflife_te, embeddings in ..\stage3\embeddings\mrnabench


---

## Part 1: inspect dataset suitability

A transcript contains a coding sequence (`CDS`), which specifies the translated protein, and can include untranslated regions before and after it: the 5′ UTR and 3′ UTR. Those untranslated regions remain part of the RNA. A region comparison needs their boundaries as well as a target and suitable evaluation groups.

These files supply a codon-start track rather than a simple start/end coordinate pair. The helper locates its marked positions and checks that successive marks are three nucleotides apart, then slices the transcript into regions. This checks the annotation's spacing; it does not experimentally establish translation or fully validate every terminal codon. Rows whose tracks cannot be interpreted retain a `frame_problem` reason for later filtering.

The table combines file-derived counts with catalog metadata. Region availability, lengths and gene counts come from the loaded rows; species, target descriptions and intended split rules come from the configured catalog. Missing dataset files are downloaded into the ignored data cache. Saved timings describe those earlier loads, not the duration of a future run.


In [3]:
gate_rows, frames = [], {}
for key, spec in mrnabench.DATASETS.items():
    started = time.perf_counter()
    frames[key] = mrnabench.load(key, DATA_CACHE)
    gate_rows.append(mrnabench.gate(frames[key], spec))
    print(f"  {key}: {len(frames[key]):,} rows in {time.perf_counter() - started:.0f} s")

gate = pd.DataFrame(gate_rows).set_index("dataset")
display(
    gate[[
        "rows", "species", "targets present", "rows with a coding region",
        "rows with a 5' region", "rows with a 3' region", "median transcript nt",
        "longest transcript nt", "distinct genes",
    ]].style.format({
        "rows": "{:,}",
        "rows with a coding region": "{:.1%}",
        "rows with a 5' region": "{:.1%}",
        "rows with a 3' region": "{:.1%}",
        "median transcript nt": "{:,}",
        "longest transcript nt": "{:,}",
        "distinct genes": "{:,}",
    })
)


  rnahl-human: 12,939 rows in 144 s
  rnahl-mouse: 13,723 rows in 33 s
  translation-efficiency-human: 10,281 rows in 20 s
  translation-efficiency-mouse: 10,853 rows in 38 s
  mrl-sugimoto: 11,693 rows in 17 s
  mrl-hl-lbkwk: 203 rows in 1 s


,rows,species,targets present,rows with a coding region,rows with a 5' region,rows with a 3' region,median transcript nt,longest transcript nt,distinct genes
dataset,,,,,,,,,
rnahl-human,"12,939",human,target 100%,100.0%,100.0%,99.6%,"3,105","12,288","12,939"
rnahl-mouse,"13,723",mouse,target 100%,100.0%,100.0%,99.7%,"2,872","12,288","13,723"
translation-efficiency-human,"10,281",human,target 100%,100.0%,100.0%,100.0%,"3,391","17,497","10,280"
translation-efficiency-mouse,"10,853",mouse,target 100%,100.0%,100.0%,100.0%,"3,109","12,252","10,848"
mrl-sugimoto,"11,693",human,target 100%,84.9%,98.6%,89.4%,"1,976","12,275","7,392"
mrl-hl-lbkwk,203,synthetic,"target_in_cell_half_life 100%, target_ribosome_load 100%",100.0%,100.0%,100.0%,925,"1,585",0


**Reading the checks.** The coding-region fraction uses all rows as its denominator. The 5′- and 3′-region fractions are calculated among rows with an interpretable coding track. A dataset-level “passes” verdict does not mean that every transcript has both UTRs, every target is complete, or the eventual split is free of sequence or gene-family overlap. The verdicts and caveats below should be read with the coverage table.


In [4]:
for key, row in gate.iterrows():
    print(f"{key}\n    verdict: {row['verdict']}")
    if row["caveats"]:
        print(f"    watch:   {row['caveats']}")
    print(f"    measures: {row['measures']}")

# Save the full inspection table, including fields not shown in the compact display.
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
gate.to_csv(RESULTS_DIR / "gate.csv")


rnahl-human
    verdict: passes: full transcript, regions and grouped splits all present
    watch:   0.4% of coding rows have no 3' region
    measures: transcript half-life, first principal component over 39 human datasets
rnahl-mouse
    verdict: passes: full transcript, regions and grouped splits all present
    watch:   0.3% of coding rows have no 3' region
    measures: transcript half-life, first principal component over 27 mouse datasets
translation-efficiency-human
    verdict: passes: full transcript, regions and grouped splits all present
    measures: translation efficiency from ribosome profiling, one transcript per gene
translation-efficiency-mouse
    verdict: passes: full transcript, regions and grouped splits all present
    measures: translation efficiency from ribosome profiling, one transcript per gene
mrl-sugimoto
    verdict: passes: full transcript, regions and grouped splits all present
    watch:   15.1% of rows mark no coding region, so a region comparison mus

### Select the datasets for region-feature comparisons

Part 2 uses datasets whose verdict starts with `passes` and whose supplied table has at least `MINIMUM_ROWS` rows. That existing 1,000-row threshold is a screening choice; it does not establish statistical power for a particular effect. The synthetic-mRNA table appears in the retained inspection but is held out of the retained comparison because of its size.

Within each selected dataset, later calculations exclude rows with a nonempty `frame_problem`. Such a row lacks a usable coding-track interpretation; that alone does not establish that it is biologically noncoding. Rows with an interpretable track can still lack a UTR, and the filter does not enforce complete terminal codons. These limits matter when calling an analysis a “full-transcript” comparison.


In [5]:
usable = [
    key
    for key, row in gate.iterrows()
    if row["verdict"].startswith("passes")
    and gate.loc[key, "rows"] >= mrnabench.MINIMUM_ROWS
]
print("datasets that pass the gate and are large enough:")
for key in usable:
    print(f"  {key}: {mrnabench.DATASETS[key].measures}")

held_back = [k for k in mrnabench.DATASETS if k not in usable]
print("\nheld back from the headline comparison:")
for key in held_back:
    print(f"  {key}: {gate.loc[key, 'verdict']} | {gate.loc[key, 'caveats']}")


datasets that pass the gate and are large enough:
  rnahl-human: transcript half-life, first principal component over 39 human datasets
  rnahl-mouse: transcript half-life, first principal component over 27 mouse datasets
  translation-efficiency-human: translation efficiency from ribosome profiling, one transcript per gene
  translation-efficiency-mouse: translation efficiency from ribosome profiling, one transcript per gene
  mrl-sugimoto: mean ribosome load per transcript isoform, NOT half-life

held back from the headline comparison:
  mrl-hl-lbkwk: passes: full transcript, regions and grouped splits all present | 203 rows is too few to resolve small differences; the library is synthetic, so rows are split at random and a gene grouping does not apply


---

## Part 2: compare features calculated from transcript regions

Before loading an encoder, test how much the target can be predicted from sequence length and composition. Each requested region contributes its GC fraction and `log1p` of its length. The CDS additionally contributes the shared 87-feature vector: GC, GC at the third codon position, log length, 64 codon frequencies and 20 amino-acid frequencies. That inherited construction repeats CDS GC and includes both log(length) and log1p(length); this comparison retains those features.

“Full transcript” here means combining the features calculated separately for the 5′ UTR, CDS and 3′ UTR. Codon counts come from the CDS, not from triplets starting arbitrarily in a UTR. The CDS → CDS plus 3′ UTR → full-transcript conditions form a nested feature comparison; the UTR-only conditions ask separate questions about each untranslated region. Empty regions contribute zero GC and log-length values.

### Define the region sets

Keep the compared regions together so the feature definition is visible before fitting. The small example below reports feature widths, not predictive performance. [The region-feature helper](../shared_code/mrnabench.py) owns the calculation.


In [ ]:
REGION_SETS = {
    "CDS only": ("cds",),
    "CDS + 3' UTR": ("cds", "utr3"),
    "5' UTR only": ("utr5",),
    "3' UTR only": ("utr3",),
    "full transcript": ("utr5", "cds", "utr3"),
}


In [6]:
print("features per set, for one example row:")
example = mrnabench.region_frame(mrnabench.load(PRIMARY[0], DATA_CACHE).head(2))
for name, parts in REGION_SETS.items():
    print(f"  {name:<18} {mrnabench.region_features(example, parts).shape[1]} features")


features per set, for one example row:
  CDS only           89 features
  CDS + 3' UTR       91 features
  5' UTR only        2 features
  3' UTR only        2 features
  full transcript    93 features


### Fit and evaluate the region baselines

The helper assigns whole groups to training, validation and test partitions, targeting 70%, 15% and 15% of rows. Group sizes can shift those proportions. When a homology map is available, related genes can share a group; otherwise the helper groups by gene. The retained results used the gene fallback, which does not ensure separation of related gene families or identical sequences assigned to different genes. The printed grouping description identifies which rule a run actually uses.

For each region set, a fresh pipeline fits feature scaling and ridge regression on training rows. `RidgeCV` chooses among five penalties internally; the outer validation partition is scored but does not select that penalty. The internal penalty selection is not supplied with the outer group labels. See the [linear-probing guide](../../docs/methods/linear-probing.md) for the role of the predictor.

Part 2 repeats this procedure for the **first three split seeds**. `test r` is mean test Pearson correlation, measuring linear association between predictions and targets; `spread` is the standard deviation across those three assignments. The same dataset rows and grouping rule are reused across its region sets.


In [7]:
baseline_rows = []
for key in usable:
    spec = mrnabench.DATASETS[key]
    regions = mrnabench.region_frame(frames[key])

    # Retain rows with an interpretable coding track, in their original order.
    keep = regions["frame_problem"] == ""
    regions = regions[keep].reset_index(drop=True)
    groups, grouping_described = mrnabench.grouping(regions, spec.species, DATA_CACHE)
    print(
        f"\n{key}: {len(regions):,} rows with a reading frame, "
        f"grouped by {grouping_described}"
    )

    for target in spec.targets:
        y = regions[target].to_numpy(dtype=float)
        for name, parts in REGION_SETS.items():
            X = mrnabench.region_features(regions, parts)
            scores = mrnabench.probe_over_seeds(X, y, groups, PROBE_SEEDS[:3])
            baseline_rows.append({
                "dataset": key,
                "target": target,
                "representation": name,
                "test r": scores["test_r"].mean(),
                "spread": scores["test_r"].std(),
                "seeds": len(scores),
            })
            print(
                f"    {name:<18} Pearson r {scores['test_r'].mean():.3f} "
                f"(+/- {scores['test_r'].std():.3f} over {len(scores)} splits)"
            )

baselines = pd.DataFrame(baseline_rows)
baselines.to_csv(RESULTS_DIR / "composition_baselines.csv", index=False)
display(baselines.pivot_table(
    index=["dataset", "target"],
    columns="representation",
    values="test r",
).round(3))



rnahl-human: 12,939 rows with a reading frame, grouped by gene (the homology map is not downloaded; weaker than the benchmark's)
    CDS only           Pearson r 0.487 (+/- 0.003 over 3 splits)
    CDS + 3' UTR       Pearson r 0.534 (+/- 0.007 over 3 splits)
    5' UTR only        Pearson r 0.177 (+/- 0.017 over 3 splits)
    3' UTR only        Pearson r 0.260 (+/- 0.018 over 3 splits)
    full transcript    Pearson r 0.542 (+/- 0.016 over 3 splits)

rnahl-mouse: 13,723 rows with a reading frame, grouped by gene (the homology map is not downloaded; weaker than the benchmark's)
    CDS only           Pearson r 0.447 (+/- 0.021 over 3 splits)
    CDS + 3' UTR       Pearson r 0.487 (+/- 0.023 over 3 splits)
    5' UTR only        Pearson r 0.173 (+/- 0.010 over 3 splits)
    3' UTR only        Pearson r 0.254 (+/- 0.022 over 3 splits)
    full transcript    Pearson r 0.498 (+/- 0.020 over 3 splits)

translation-efficiency-human: 10,277 rows with a reading frame, grouped by gene (the homo

,representation,3' UTR only,5' UTR only,CDS + 3' UTR,CDS only,full transcript
dataset,target,,,,,
mrl-sugimoto,target,0.031,0.035,0.198,0.200,0.199
rnahl-human,target,0.260,0.177,0.534,0.487,0.542
rnahl-mouse,target,0.254,0.173,0.487,0.447,0.498
translation-efficiency-human,target,0.135,0.280,0.685,0.683,0.702
translation-efficiency-mouse,target,0.088,0.234,0.632,0.629,0.660


**What the retained baselines show.** For the human and mouse half-life and translation-efficiency tables, the saved full-transcript feature scores exceed the CDS-only feature scores. The ribosome-load table shows little corresponding difference. These observations concern the selected features and grouped ridge procedure; they do not establish a biological effect of adding a particular region. Part 3 now asks what the pretrained representations contribute on the two selected human datasets.


---

## Part 3: frozen encoder representations and their combinations

The human half-life and translation-efficiency comparisons use the same three encoder identities as BioLangFusion: NT v2 100M, RNA-FM and ESM-2 8M. Their inputs and ridge predictor differ from Phase A's token-level experiment. RNA-FM reads selected transcript regions; NT reads the supplied transcript in its DNA-letter alphabet; ESM reads the CDS translation. The DNA encoder receives no genomic flanking sequence here.

The encoder conditions include the 5′ UTR, CDS, full transcript and protein, but do not include a CDS-plus-3′-UTR encoder condition. That condition appears only in the calculated-feature comparison above. The settings below define exactly which inputs are extracted.

### Define encoder inputs and window sizes

Long inputs are divided into consecutive, nonoverlapping character windows chosen from each encoder's configured token limit. Each window is encoded independently, and the shared helper concatenates and averages the returned token states, including special-token states such as sequence markers. This represents sequence across the supplied regions without modeling interactions between windows in that pass. Window sizing and tokenizer truncation also limit what can be claimed about complete input coverage.

**Eligibility limitation.** The check below prints a “skips it” message for an ineligible `PRIMARY` entry, but the later extraction and scoring loops still iterate over `PRIMARY` directly. It does not enforce that skip. The retained human datasets passed the screen; a changed dataset selection needs this discrepancy resolved before relying on automatic exclusion.


In [ ]:
# One encoder for each representation type, retaining the BioLangFusion selections.
STREAM_ENCODERS = {
    "dna": "ntv2_100m",
    "rna": "rnafm",
    "prot": "esm2_8m",
}

# Each entry selects an encoder type and the transcript regions it receives.
INPUTS = {
    "5' UTR": ("rna", ("utr5",)),
    "CDS": ("rna", ("cds",)),
    "full transcript": ("rna", ("utr5", "cds", "utr3")),
    "full transcript, DNA encoder": ("dna", ("utr5", "cds", "utr3")),
    "protein of the CDS": ("prot", ("cds",)),
}


In [9]:
# Informational only: this loop does not filter PRIMARY for the later cells.
for key in PRIMARY:
    if key not in usable:
        print(f"{key} did not pass the gate; Part 3 skips it.")

# Characters are nucleotides for DNA/RNA inputs and residues for protein inputs.
print(f"\nwindows per encoder: " + ", ".join(
    f"{k}: {mrnabench.default_window(ENCODERS[v]):,} characters"
    for k, v in STREAM_ENCODERS.items()
))



windows per encoder: dna: 12,276 characters, rna: 1,022 characters, prot: 1,022 characters


### Convert each region selection into encoder inputs

For a nucleotide encoder, join the selected regions in transcript order and convert letters to the encoder's accepted alphabet. For the protein encoder, translate the CDS using the shared translation helper, which stops at the first stop codon. The resulting list keeps the filtered dataset's row order, so each embedding can later be paired with the corresponding target.


In [10]:
def stream_texts(regions, modality, parts):
    encoder = ENCODERS[STREAM_ENCODERS[modality]]
    if modality == "prot":
        # Translate the coding region rather than untranslated transcript sequence.
        return [
            mrnabench.encoder_text(encoder, row["cds"])
            for _, row in regions.iterrows()
        ]

    joined = ["".join(row[p] for p in parts) for _, row in regions.iterrows()]
    return [mrnabench.encoder_text(encoder, text) for text in joined]


### Load or extract complete representation matrices

This cell is where encoder work begins; there is no preceding timing trial. A compatible complete cache can be loaded without running the encoder. Cache checks include the requested encoder, checkpoint/revision, window size, row count and ordered-input fingerprint. A mismatch is an error rather than permission to overwrite an unrelated matrix.

For a missing cache, the helper processes all inputs for that representation and saves the complete matrix afterward. Messages every 200 inputs report progress, not checkpoints. Cancellation before publication loses that matrix's partial computation; previously completed matrices can still be reused. The printed elapsed time includes the work performed by that call and is a past observation, not an estimate of time remaining.


In [11]:
if RUN_EMBEDDINGS:
    embedded = {}
    for key in PRIMARY:
        spec = mrnabench.DATASETS[key]
        regions = mrnabench.region_frame(frames[key])
        regions = regions[regions["frame_problem"] == ""].reset_index(drop=True)

        for name, (modality, parts) in INPUTS.items():
            texts = stream_texts(regions, modality, parts)
            folder = EMBED_DIR / key
            tag = name.replace("'", "").replace(" ", "_").replace(",", "")
            started = time.perf_counter()
            matrix, reused = mrnabench.embed_with_cache(
                STREAM_ENCODERS[modality],
                texts,
                folder / f"{tag}.npz",
                device=DEVICE,
            )
            embedded[(key, name)] = matrix
            print(
                f"  {key} / {name}: {matrix.shape}, "
                f"{'reused' if reused else f'{(time.perf_counter() - started) / 60:.1f} min'}"
            )
else:
    print("RUN_EMBEDDINGS is off. Part 1 and Part 2 run without it; turn it on for a GPU night.")


W1022 05:32:32.914000 8888 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.dense.weight      | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
pooler.dense.bias                   | MISSING    | 
pooler.dense.weight                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    rnafm: 200/12939
    rnafm: 400/12939
    rnafm: 600/12939
    rnafm: 800/12939
    rnafm: 1000/12939
    rnafm: 1200/12939
    rnafm: 1400/12939
    rnafm: 1600/12939
    rnafm: 1800/12939
    rnafm: 2000/12939
    rnafm: 2200/12939
    rnafm: 2400/12939
    rnafm: 2600/12939
    rnafm: 2800/12939
    rnafm: 3000/12939
    rnafm: 3200/12939
    rnafm: 3400/12939
    rnafm: 3600/12939
    rnafm: 3800/12939
    rnafm: 4000/12939
    rnafm: 4200/12939
    rnafm: 4400/12939
    rnafm: 4600/12939
    rnafm: 4800/12939
    rnafm: 5000/12939
    rnafm: 5200/12939
    rnafm: 5400/12939
    rnafm: 5600/12939
    rnafm: 5800/12939
    rnafm: 6000/12939
    rnafm: 6200/12939
    rnafm: 6400/12939
    rnafm: 6600/12939
    rnafm: 6800/12939
    rnafm: 7000/12939
    rnafm: 7200/12939
    rnafm: 7400/12939
    rnafm: 7600/12939
    rnafm: 7800/12939
    rnafm: 8000/12939
    rnafm: 8200/12939
    rnafm: 8400/12939
    rnafm: 8600/12939
    rnafm: 8800/12939
    rnafm: 9000/12939
    rnafm: 920

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.dense.weight      | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
pooler.dense.bias                   | MISSING    | 
pooler.dense.weight                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    rnafm: 200/10277
    rnafm: 400/10277
    rnafm: 600/10277
    rnafm: 800/10277
    rnafm: 1000/10277
    rnafm: 1200/10277
    rnafm: 1400/10277
    rnafm: 1600/10277
    rnafm: 1800/10277
    rnafm: 2000/10277
    rnafm: 2200/10277
    rnafm: 2400/10277
    rnafm: 2600/10277
    rnafm: 2800/10277
    rnafm: 3000/10277
    rnafm: 3200/10277
    rnafm: 3400/10277
    rnafm: 3600/10277
    rnafm: 3800/10277
    rnafm: 4000/10277
    rnafm: 4200/10277
    rnafm: 4400/10277
    rnafm: 4600/10277
    rnafm: 4800/10277
    rnafm: 5000/10277
    rnafm: 5200/10277
    rnafm: 5400/10277
    rnafm: 5600/10277
    rnafm: 5800/10277
    rnafm: 6000/10277
    rnafm: 6200/10277
    rnafm: 6400/10277
    rnafm: 6600/10277
    rnafm: 6800/10277
    rnafm: 7000/10277
    rnafm: 7200/10277
    rnafm: 7400/10277
    rnafm: 7600/10277
    rnafm: 7800/10277
    rnafm: 8000/10277
    rnafm: 8200/10277
    rnafm: 8400/10277
    rnafm: 8600/10277
    rnafm: 8800/10277
    rnafm: 9000/10277
    rnafm: 920

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    esm2_8m: 200/10277
    esm2_8m: 400/10277
    esm2_8m: 600/10277
    esm2_8m: 800/10277
    esm2_8m: 1000/10277
    esm2_8m: 1200/10277
    esm2_8m: 1400/10277
    esm2_8m: 1600/10277
    esm2_8m: 1800/10277
    esm2_8m: 2000/10277
    esm2_8m: 2200/10277
    esm2_8m: 2400/10277
    esm2_8m: 2600/10277
    esm2_8m: 2800/10277
    esm2_8m: 3000/10277
    esm2_8m: 3200/10277
    esm2_8m: 3400/10277
    esm2_8m: 3600/10277
    esm2_8m: 3800/10277
    esm2_8m: 4000/10277
    esm2_8m: 4200/10277
    esm2_8m: 4400/10277
    esm2_8m: 4600/10277
    esm2_8m: 4800/10277
    esm2_8m: 5000/10277
    esm2_8m: 5200/10277
    esm2_8m: 5400/10277
    esm2_8m: 5600/10277
    esm2_8m: 5800/10277
    esm2_8m: 6000/10277
    esm2_8m: 6200/10277
    esm2_8m: 6400/10277
    esm2_8m: 6600/10277
    esm2_8m: 6800/10277
    esm2_8m: 7000/10277
    esm2_8m: 7200/10277
    esm2_8m: 7400/10277
    esm2_8m: 7600/10277
    esm2_8m: 7800/10277
    esm2_8m: 8000/10277
    esm2_8m: 8200/10277
    esm2_8m: 8400/10

### Compare individual and concatenated representations

Each cached matrix contains one pooled vector per retained transcript row. `full transcript + protein` joins the RNA-FM transcript vector with the ESM protein vector for that same row. `three encoders fused` additionally includes the NT transcript vector. `np.hstack` joins their feature columns; it does not learn a fusion layer or update encoder weights.

The same grouped ridge procedure now uses **all nine split seeds** for every representation in each human dataset. These are different held-out assignments, unlike IsoFormer's repeated training runs with a fixed test partition. `test r` is mean Pearson correlation, and `test Spearman` summarizes rank agreement. The saved `standard error` is the standard deviation of split scores divided by the square root of the number of assignments. It is not a paired confidence interval for a difference between representations.

The CSV stores aggregate scores and grouping descriptions. It does not retain per-split predictions or fitted ridge coefficients for a later paired analysis.


In [12]:
if RUN_EMBEDDINGS and embedded:
    rows_out = []
    for key in PRIMARY:
        spec = mrnabench.DATASETS[key]
        regions = mrnabench.region_frame(frames[key])
        regions = regions[regions["frame_problem"] == ""].reset_index(drop=True)
        groups, described = mrnabench.grouping(regions, spec.species, DATA_CACHE)

        for target in spec.targets:
            y = regions[target].to_numpy(dtype=float)
            available = {
                name: embedded[(key, name)]
                for dataset_key, name in embedded
                if dataset_key == key
            }
            available["full transcript + protein"] = np.hstack([
                available["full transcript"],
                available["protein of the CDS"],
            ])
            available["three encoders fused"] = np.hstack([
                available["full transcript, DNA encoder"],
                available["full transcript"],
                available["protein of the CDS"],
            ])

            for name, X in available.items():
                scores = mrnabench.probe_over_seeds(X, y, groups, PROBE_SEEDS)
                rows_out.append({
                    "dataset": key,
                    "target": target,
                    "representation": name,
                    "test r": scores["test_r"].mean(),
                    "standard error": scores["test_r"].std() / np.sqrt(len(scores)),
                    "test Spearman": scores["test_spearman"].mean(),
                    "seeds": len(scores),
                    "grouping": described,
                })
                print(f"  {key} / {name}: r {scores['test_r'].mean():.3f}")

    encoder_scores = pd.DataFrame(rows_out)
    encoder_scores.to_csv(RESULTS_DIR / "encoder_probes.csv", index=False)
    display(encoder_scores.pivot_table(
        index=["dataset", "target"],
        columns="representation",
        values="test r",
    ).round(3))
else:
    print("Nothing to score yet.")


  rnahl-human / 5' UTR: r 0.178
  rnahl-human / CDS: r 0.473
  rnahl-human / full transcript: r 0.512
  rnahl-human / full transcript, DNA encoder: r 0.514
  rnahl-human / protein of the CDS: r 0.552
  rnahl-human / full transcript + protein: r 0.606
  rnahl-human / three encoders fused: r 0.602
  translation-efficiency-human / 5' UTR: r 0.433
  translation-efficiency-human / CDS: r 0.630
  translation-efficiency-human / full transcript: r 0.610
  translation-efficiency-human / full transcript, DNA encoder: r 0.660
  translation-efficiency-human / protein of the CDS: r 0.682
  translation-efficiency-human / full transcript + protein: r 0.712
  translation-efficiency-human / three encoders fused: r 0.713


,representation,5' UTR,CDS,full transcript,full transcript + protein,"full transcript, DNA encoder",protein of the CDS,three encoders fused
dataset,target,,,,,,,
rnahl-human,target,0.178,0.473,0.512,0.606,0.514,0.552,0.602
translation-efficiency-human,target,0.433,0.630,0.610,0.712,0.660,0.682,0.713


**Reading the retained encoder results.** The protein-only representation has the highest saved single-encoder mean for both human targets, and the concatenated representations have higher means still. Adding the DNA-encoder vector to transcript plus protein does not consistently improve the saved scores: the three-vector mean is lower for half-life and slightly higher for translation efficiency. The table alone does not establish uncertainty in those differences.

### Calculate descriptive gains over the selected baselines

`delta fusion` subtracts the largest mean **test** Pearson correlation among three single-encoder candidates: the RNA full transcript, DNA full transcript and CDS-derived protein. The protein candidate does not receive the full transcript, so this is the strongest of the selected single representations rather than a strictly full-transcript-only baseline. Selection by observed test means is part of the retained procedure.

`over composition` subtracts the full-transcript calculated-feature mean from Part 2. That compares a nine-assignment encoder mean with a three-assignment composition mean. Both columns are descriptive differences; neither is a paired uncertainty estimate or a residualized test of information beyond composition.


In [13]:
if RUN_EMBEDDINGS and "encoder_scores" in dir() and len(encoder_scores):
    gains = []
    for (key, target), block in encoder_scores.groupby(["dataset", "target"]):
        by_name = block.set_index("representation")["test r"]

        # This candidate set includes the CDS-derived protein representation.
        singles = [
            n
            for n in (
                "full transcript",
                "full transcript, DNA encoder",
                "protein of the CDS",
            )
            if n in by_name
        ]
        best_single = by_name[singles].max()
        floor = baselines.query(
            "dataset == @key and target == @target and "
            "representation == 'full transcript'"
        )["test r"]

        for fused in ("full transcript + protein", "three encoders fused"):
            if fused in by_name:
                gains.append({
                    "dataset": key,
                    "target": target,
                    "fused model": fused,
                    "fused r": by_name[fused],
                    "best single r": best_single,
                    "delta fusion": by_name[fused] - best_single,
                    "over composition": (
                        by_name[fused] - (floor.iloc[0] if len(floor) else np.nan)
                    ),
                })

    display(pd.DataFrame(gains).round(3))
    pd.DataFrame(gains).to_csv(RESULTS_DIR / "fusion_gains.csv", index=False)


,dataset,target,fused model,fused r,best single r,delta fusion,over composition
0,rnahl-human,target,full transcript + protein,0.606,0.552,0.053,0.064
1,rnahl-human,target,three encoders fused,0.602,0.552,0.050,0.060
2,translation-efficiency-human,target,full transcript + protein,0.712,0.682,0.030,0.010
3,translation-efficiency-human,target,three encoders fused,0.713,0.682,0.031,0.012


---

## Findings, limitations and what remains

The retained region-feature results support examining untranslated regions as well as the CDS on the half-life and translation-efficiency tables. They do not imply that every region or encoder benefits equally. The separate ribosome-load comparison illustrates why the target and dataset must remain explicit.

For the two human encoder comparisons, joining the RNA full-transcript and protein vectors improves mean test Pearson correlation over the strongest selected single encoder. The [October 22 result record](../../docs/run-records.md#2026-10-22-stage4-human-probes-completed-half-life-translation-efficiency-and-lucaone-results) reports gains of about 0.053 for half-life and 0.030 for translation efficiency. These are predictive gains under this frozen, window-pooled ridge procedure. They do not establish an independent biological mechanism, a significant paired improvement, or the benefit of a trained cross-attention architecture.

The retained evaluation groups by gene, which is weaker than holding out related gene families. Windowed encoding cannot capture interactions across window boundaries in one pass, composition and encoder averages use different numbers of assignments, and the best single representation is selected from test means. These limits prevent treating the table as an exact reproduction of mRNABench, Saluki or RiboNN, or as a controlled comparison with the earlier BioLangFusion dataset.

The current encoder selection covers the human pair; mouse encoder replication is a separate scope. Before extending the comparisons, the dataset gate's enforcement and validation limits need explicit attention. Paired uncertainty would require a suitable retained or newly authorized per-split analysis. The [Stage 4 guide](README.md#full-transcript-half-life-and-translation-efficiency) connects these findings to the broader choice of representations and fusion methods.
